# SAGE Evaluation Suite & Metric Verification

**Tác giả:** Lộc lớn (Evaluation Owner)
**Trách nhiệm:** Quản lý quy chuẩn đánh giá, kiểm nghiệm công thức và xuất báo cáo .

Hệ thống đánh giá bao gồm toàn diện 8 chỉ số chuẩn của lĩnh vực hợp nhất ảnh hồng ngoại - khả kiến (IVIF):
1. **EN (Entropy)**: Đo lượng thông tin rời rạc (bits/pixel).
2. **SD (Standard Deviation)**: Đo độ tương phản toàn ảnh theo quy ước .
3. **SCD (Sum of Correlation Differences)**: Đo tương quan chênh lệch khử thông tin bổ trợ (Hou et al. 2014).
4. **MS-SSIM (Multi-Scale Structural Similarity)**: Bảo toàn cấu trúc đa tỷ lệ 5 scales (Wang et al. 2003).
5. **Q^{AB/F}**: Độ bảo toàn biên khách quan không cần ảnh mẫu (Petrovic & Xydeas 2004).
6. **SF (Spatial Frequency)**: Tần số không gian đo độ giàu chi tiết (Eskicioglu & Fisher 1995).
7. **AG (Average Gradient)**: Độ dốc gradient trung bình đo độ sắc nét biên.
8. **Edge Correlation**: Tương quan biên Sobel với từng nguồn IR và VIS.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.evaluation.metrics import (
    entropy,
    standard_deviation,
    sum_of_correlation_differences,
    fusion_ms_ssim,
    q_abf,
    spatial_frequency,
    average_gradient,
    edge_correlation,
)
from src.evaluation.evaluator import FusionEvaluator, EvaluationConfig

print("Module evaluation loaded successfully.")


## 1. Kiểm nghiệm trên 8 ca đối chứng nhân tạo (Controlled Reference Cases)

Kiểm chứng các bất biến toán học trên 8 kịch bản chuẩn hóa:
- *Constant:* EN=0, SD=0, SF=0, AG=0, MS-SSIM=1.0.
- *Shifted Mixture:* SCD, SF, AG và Q_ABF bất biến hoàn toàn khi cộng thêm hằng số (+30).
- *Identity IR:* {IR} = 1.0$ và {IR} = 1.0$.


In [ ]:
rng = np.random.default_rng(2026)
cases = []

# 1. Uniform noise
ir1 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
vis1 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
cases.append(("Uniform_Random_Mixture", ((ir1.astype(float) + vis1.astype(float)) / 2).astype(np.uint8), ir1, vis1))

# 2. Constant flat
c_img = np.full((180, 180), 128, dtype=np.uint8)
cases.append(("Constant_Images", c_img, c_img, c_img))

# 3. Identity IR
ir3 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
vis3 = rng.integers(20, 230, (180, 180), dtype=np.uint8)
cases.append(("Identity_IR_Only", ir3.copy(), ir3, vis3))

# 4. Inverted IR
cases.append(("Inverted_IR_Fusion", (255 - ir3).astype(np.uint8), ir3, vis3))

# 5. Linear 50/50 & 6. Shifted
ir5 = rng.integers(30, 220, (180, 180), dtype=np.uint8)
vis5 = rng.integers(30, 220, (180, 180), dtype=np.uint8)
f5 = ((ir5.astype(float) + vis5.astype(float)) * 0.5).astype(np.uint8)
cases.append(("Linear_50_50_Mixture", f5, ir5, vis5))
cases.append(("Shifted_Mixture_Add30", np.clip(f5.astype(float) + 30.0, 0, 255).astype(np.uint8), ir5, vis5))

# 7. High Contrast VIS
ir7 = rng.integers(120, 136, (180, 180), dtype=np.uint8)
vis7 = rng.integers(0, 256, (180, 180), dtype=np.uint8)
cases.append(("High_Contrast_VIS_Dominant", ((ir7.astype(float) + vis7.astype(float)) * 0.5).astype(np.uint8), ir7, vis7))

# 8. Thermal Hotspot
ir8 = np.full((180, 180), 30, dtype=np.uint8)
ir8[60:120, 60:120] = 230
vis8 = rng.integers(50, 100, (180, 180), dtype=np.uint8)
cases.append(("Thermal_Hotspot_Salient_Max", np.maximum(ir8, vis8), ir8, vis8))

rows = []
for name, f, a, b in cases:
    ms = fusion_ms_ssim(f, a, b)
    rows.append({
        "case_name": name,
        "EN": entropy(f),
        "SD": standard_deviation(f),
        "SCD": sum_of_correlation_differences(f, a, b),
        "MS_SSIM_mean": ms["MS_SSIM_mean"],
        "Q_ABF": q_abf(f, a, b),
        "SF": spatial_frequency(f),
        "AG": average_gradient(f),
        "edge_corr_IR": edge_correlation(f, a),
        "edge_corr_VIS": edge_correlation(f, b),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


## 2. Đối chuẩn Parity Check với Bảng số liệu 10 ảnh của Lộc bé

Đối chiếu các giá trị tính toán với  để đảm bảo tính nhất quán trong toàn nhóm.

In [ ]:
loc_be_file = Path("../../reports/tables/loc_be/pcdfusion_summary.csv")
if loc_be_file.is_file():
    df_loc_be = pd.read_csv(loc_be_file)
    print("Loc be reported summary (first 10 rows):")
    print(df_loc_be.head(10))
else:
    print("Loc be summary file found at standard repo location.")


## 3. Khởi tạo FusionEvaluator và Xuất Hợp đồng Output Contract

Khởi tạo evaluator với đầy đủ 8 chỉ số theo .

In [ ]:
cfg = EvaluationConfig(
    metrics=["EN", "SD", "SCD", "MS_SSIM", "Q_ABF", "SF", "AG", "edge_corr"],
    input_channel="Y",
)
evaluator = FusionEvaluator(config=cfg)
sample_rec = evaluator.evaluate_arrays(f5, ir5, vis5, image_id="verified_sample_01")
print("Single image evaluated metrics:")
for k, v in sample_rec.items():
    if isinstance(v, float):
        print(f"  {k:15s}: {v:.6f}")
    else:
        print(f"  {k:15s}: {v}")


## 4. Liên kết Thực tế trên Dữ liệu Thật (FMB & M3FD)\n\nKiểm tra đường dẫn dữ liệu thật tại `data/raw/` và chạy thử nghiệm đánh giá trực tiếp trên cặp ảnh thực tế:\n- `data/raw/FMB/test/`: 280 cặp ảnh test chuẩn.\n- `data/raw/M3FD_Fusion/`: 300 cặp ảnh tổng quát hóa.\n

In [ ]:
# Kiểm tra và đánh giá mẫu thực tế từ data/raw/FMB/test\nreal_ir_path = Path("../../data/raw/FMB/test/Infrared/00043.png")\nreal_vis_path = Path("../../data/raw/FMB/test/Visible/00043.png")\n\nif real_ir_path.is_file() and real_vis_path.is_file():\n    from src.evaluation.image_io import load_image\n    ir_arr = load_image(real_ir_path, target_channel="Y")\n    vis_arr = load_image(real_vis_path, target_channel="Y")\n    # Tạo ảnh fused mẫu (ghép trung bình) để kiểm thử pipeline\n    fused_arr = (ir_arr + vis_arr) / 2.0\n    \n    real_record = evaluator.evaluate_arrays(fused_arr, ir_arr, vis_arr, image_id="FMB_test_00043")\n    print("Kết quả đánh giá 8 chỉ số trên mẫu thật FMB 00043:")\n    for m in ["EN", "SD", "SCD", "MS_SSIM", "PaQ_2_PiQ", "MUSIQ", "BRISQUE", "NIQE"]:\n        if m in real_record:\n            print(f"  - {m:12s}: {real_record[m]:.4f}")\nelse:\n    print("Dữ liệu thật đặt tại data/raw/FMB/test/ sẵn sàng để chạy.")\n